In [1]:
# imports
import tensorflow as tf
from tensorflow import keras as keras
import matplotlib.pyplot as plt
import keras_tuner as kt

#tf.keras.utils.set_random_seed(1032)
print(tf.config.list_physical_devices('GPU'))

2024-05-21 14:08:10.817563: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:9261] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2024-05-21 14:08:10.817616: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:607] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2024-05-21 14:08:10.818650: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1515] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2024-05-21 14:08:10.824935: I tensorflow/core/platform/cpu_feature_guard.cc:182] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2024-05-21 14:08:11.466685: W tensorflow/compiler/tf2

[PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


2024-05-21 14:08:12.130157: I external/local_xla/xla/stream_executor/cuda/cuda_executor.cc:887] could not open file to read NUMA node: /sys/bus/pci/devices/0000:01:00.0/numa_node
Your kernel may have been built without NUMA support.
2024-05-21 14:08:12.158909: I external/local_xla/xla/stream_executor/cuda/cuda_executor.cc:887] could not open file to read NUMA node: /sys/bus/pci/devices/0000:01:00.0/numa_node
Your kernel may have been built without NUMA support.
2024-05-21 14:08:12.158958: I external/local_xla/xla/stream_executor/cuda/cuda_executor.cc:887] could not open file to read NUMA node: /sys/bus/pci/devices/0000:01:00.0/numa_node
Your kernel may have been built without NUMA support.


In [2]:
#import  data

IMG_SIZE = 224
BATCH_SIZE = 32

train_data = tf.keras.utils.image_dataset_from_directory(
    "data/train",
    shuffle = True,
    image_size = (IMG_SIZE,IMG_SIZE),
    batch_size = BATCH_SIZE
)

test_data = tf.keras.utils.image_dataset_from_directory(
    directory = "data/test",
    shuffle = True,
    image_size = (IMG_SIZE, IMG_SIZE),
    batch_size = BATCH_SIZE
)

val_data = tf.keras.utils.image_dataset_from_directory(
    directory = "data/valid",
    shuffle = True,
    image_size = (IMG_SIZE,IMG_SIZE),
    batch_size = BATCH_SIZE
)

data_augmentation_layers = [
    tf.keras.layers.RandomFlip("horizontal"),
    tf.keras.layers.RandomRotation(0.1),
    tf.keras.layers.RandomZoom(0.1),
    tf.keras.layers.RandomContrast(0.1),
]

def  data_augmentation(images):
    for layer in  data_augmentation_layers:
        images = layer(images)
    return images

def preprocess_image(image, label):
    image = tf.cast(image, tf.float32) / 255.0  # Normalize pixel values to [0, 1]
    image = data_augmentation(image)
    return image, label

train_data = train_data.map(
    preprocess_image,
    num_parallel_calls=tf.data.AUTOTUNE
).prefetch(tf.data.AUTOTUNE)

val_data = val_data.map(
    lambda img, label: (tf.cast(img, tf.float32) / 255.0, label),
    num_parallel_calls=tf.data.AUTOTUNE
).prefetch(tf.data.AUTOTUNE)

test_data = test_data.map(
    lambda img, label: (tf.cast(img, tf.float32) / 255.0, label),
    num_parallel_calls=tf.data.AUTOTUNE
).prefetch(tf.data.AUTOTUNE)


Found 2339 files belonging to 10 classes.


2024-05-21 14:08:13.002207: I external/local_xla/xla/stream_executor/cuda/cuda_executor.cc:887] could not open file to read NUMA node: /sys/bus/pci/devices/0000:01:00.0/numa_node
Your kernel may have been built without NUMA support.
2024-05-21 14:08:13.002273: I external/local_xla/xla/stream_executor/cuda/cuda_executor.cc:887] could not open file to read NUMA node: /sys/bus/pci/devices/0000:01:00.0/numa_node
Your kernel may have been built without NUMA support.
2024-05-21 14:08:13.002294: I external/local_xla/xla/stream_executor/cuda/cuda_executor.cc:887] could not open file to read NUMA node: /sys/bus/pci/devices/0000:01:00.0/numa_node
Your kernel may have been built without NUMA support.
2024-05-21 14:08:13.096540: I external/local_xla/xla/stream_executor/cuda/cuda_executor.cc:887] could not open file to read NUMA node: /sys/bus/pci/devices/0000:01:00.0/numa_node
Your kernel may have been built without NUMA support.
2024-05-21 14:08:13.096592: I external/local_xla/xla/stream_executor

Found 50 files belonging to 10 classes.
Found 50 files belonging to 10 classes.


In [3]:
IMG_SIZE = 224
BATCH_SIZE = 32
CHANNELS =3
input_shape = (IMG_SIZE, IMG_SIZE, CHANNELS)

def Model(input_shape):

    model = keras.Sequential([
        keras.layers.Conv2D(32, kernel_size=3, input_shape=input_shape, activation='relu'),
        keras.layers.MaxPool2D((2,2)),
        keras.layers.BatchNormalization(),
        keras.layers.Dropout(0.2),
        
        keras.layers.Conv2D(64,kernel_size=3, activation='relu'),
        keras.layers.MaxPool2D((2,2)),
        keras.layers.BatchNormalization(),
        keras.layers.Dropout(0.2),
        
        keras.layers.Conv2D(96,kernel_size=3, activation='relu'),
        keras.layers.MaxPool2D((2,2)),
        keras.layers.BatchNormalization(),
        keras.layers.Dropout(0.2),
        
        keras.layers.Conv2D(96, kernel_size=3, activation='relu'),
        keras.layers.MaxPooling2D(pool_size=(2,2)),
        keras.layers.BatchNormalization(),
        keras.layers.Dropout(0.2),
        
        keras.layers.Conv2D(64, kernel_size=3, activation='relu'),
        keras.layers.MaxPooling2D(pool_size=(2,2)),
        keras.layers.BatchNormalization(),
        keras.layers.Dropout(0.2),
        
        
        
        keras.layers.Flatten(),
        #keras.layers.Dropout(0.25),
        keras.layers.Dense(256, activation='relu', activity_regularizer = tf.keras.regularizers.L2(0.008)),
        keras.layers.Dropout(0.25),
        keras.layers.Dense(128, activation='relu', activity_regularizer = tf.keras.regularizers.L2(0.008)),
        #keras.layers.BatchNormalization(),
        keras.layers.Dropout(0.25),
        keras.layers.Dense(10, activation='softmax', activity_regularizer = tf.keras.regularizers.L2(0.008))
    ])
    
    #, activity_regularizer = tf.keras.regularizers.L2()
    #, activity_regularizer = tf.keras.regularizers.L2(0.008)
    
    return model

Model(input_shape)

In [4]:
model = Model(input_shape)
model.compile(optimizer = keras.optimizers.Adam(learning_rate = 0.0005),
             loss = tf.keras.losses.SparseCategoricalCrossentropy(from_logits =False),
             metrics=['accuracy'])

In [5]:
earlystopping = tf.keras.callbacks.EarlyStopping(monitor="val_loss", mode="min", patience=20, restore_best_weights=True)

# min_delta=0.001
#,callbacks=[earlystopping]
history = model.fit(
train_data,
epochs = 200,
batch_size = 32,
validation_data = val_data,callbacks=[earlystopping])

Epoch 1/200


2024-05-21 14:08:21.280726: E tensorflow/core/grappler/optimizers/meta_optimizer.cc:961] layout failed: INVALID_ARGUMENT: Size of values 0 does not match size of permutation 4 @ fanin shape insequential_1/dropout_7/dropout/SelectV2-2-TransposeNHWCToNCHW-LayoutOptimizer
2024-05-21 14:08:21.933854: I external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:454] Loaded cuDNN version 8904
2024-05-21 14:08:22.098855: I external/local_tsl/tsl/platform/default/subprocess.cc:304] Start cannot spawn child process: No such file or directory
2024-05-21 14:08:23.301327: I external/local_xla/xla/service/service.cc:168] XLA service 0x7f4c711f7da0 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
2024-05-21 14:08:23.301361: I external/local_xla/xla/service/service.cc:176]   StreamExecutor device (0): NVIDIA GeForce RTX 4060 Laptop GPU, Compute Capability 8.9
2024-05-21 14:08:23.305579: I tensorflow/compiler/mlir/tensorflow/utils/dump_mlir_util.cc:269] disabling ML

74/74 [==============================] - 12s 77ms/step - loss: 3.7384 - accuracy: 0.1932 - val_loss: 2.4762 - val_accuracy: 0.1000
Epoch 2/200
74/74 [==============================] - 5s 56ms/step - loss: 2.5938 - accuracy: 0.2676 - val_loss: 2.6138 - val_accuracy: 0.1000
Epoch 3/200
74/74 [==============================] - 5s 56ms/step - loss: 2.3632 - accuracy: 0.3100 - val_loss: 2.7493 - val_accuracy: 0.1000
Epoch 4/200
74/74 [==============================] - 5s 56ms/step - loss: 2.2378 - accuracy: 0.3130 - val_loss: 2.8886 - val_accuracy: 0.1400
Epoch 5/200
74/74 [==============================] - 5s 56ms/step - loss: 2.1379 - accuracy: 0.3484 - val_loss: 2.7654 - val_accuracy: 0.1200
Epoch 6/200
74/74 [==============================] - 5s 56ms/step - loss: 2.0430 - accuracy: 0.3651 - val_loss: 2.6555 - val_accuracy: 0.1800
Epoch 7/200
74/74 [==============================] - 5s 56ms/step - loss: 1.9313 - accuracy: 0.4049 - val_loss: 2.5568 - val_accuracy: 0.2200
Epoch 8/200
74/74

KeyboardInterrupt: 